# 🎯 Feature Engineering & Modélisation
## Prédiction des Prix Immobiliers en Mauritanie

**Objectif** : Construire et évaluer plusieurs modèles de régression pour prédire le prix des biens immobiliers.

Ce notebook couvre :
1. 🔧 Feature Engineering complet
2. 📊 Préparation des données
3. 🤖 Test de 6 modèles différents
4. ✅ Validation croisée 5-fold
5. 💾 Sauvegarde du meilleur modèle

---

**Variable cible** : `prix` (en MRU - Ouguiya mauritanien)  
**Contexte** : Marché immobilier de Nouakchott

> 💡 *Projet Capstone - SupNum - Machine Learning Course - Mohamed*

## 🔧 Setup - Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# XGBoost (optionnel)
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost non disponible - installé avec: pip install xgboost")

# Configuration des graphiques
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Configuration des chemins
project_root = Path().resolve().parent
data_path = project_root / 'data' / 'raw' / 'kaggle_train.csv'
model_dir = project_root / 'model'
model_dir.mkdir(exist_ok=True)

print("✅ Librairies chargées !")
print(f"📁 Chemin des données : {data_path}")
print(f"📁 Dossier modèles : {model_dir}")

✅ Librairies chargées !
📁 Chemin des données : /home/bechir/Documents/immobilier-price-prediction/data/raw/kaggle_train.csv
📁 Dossier modèles : /home/bechir/Documents/immobilier-price-prediction/model


---
## 📥 Section 2 : Chargement des données

In [2]:
# Charger les données
df = pd.read_csv(data_path, encoding='utf-8')
print(f"📐 Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\n📊 Colonnes disponibles :")
print(list(df.columns))
print(f"\n📋 Aperçu :")
df.head()

📐 Dimensions : 1153 lignes × 12 colonnes

📊 Colonnes disponibles :
['id', 'titre', 'prix', 'surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb', 'quartier', 'description', 'caracteristiques', 'source', 'date_publication']

📋 Aperçu :


,id,titre,prix,surface_m2,nb_chambres,nb_salons,nb_sdb,quartier,description,caracteristiques,source,date_publication
0,1076,منزل احذ اللنكات حمام الياسمين,1800000.0,150.0,3.0,2.0,NaN,Arafat,دار للبيع اعل شارع اكبير احذ حمام الياسمين الل...,Titre foncier | 1 balcon(s) | Taille rue: 15.0...,voursa.com,2025-09-13
1,875,فرصة دار مكونه من طابقين ارضي و واحد فوقوني كا...,1800000.0,300.0,6.0,3.0,NaN,Tevragh Zeina,فرصة دار ، الطابق الأرضي يحتاج ترميم بسبب المل...,1 balcon(s) | Taille rue: N/A | Proche de: كرف...,voursa.com,2025-07-06
2,453,دار فتيارت فاتح فبرك,900000.0,216.0,1.0,1.0,NaN,Teyarett,سعر 9ملايين مدخله 50الف حد شاري ول عندو طلب تل...,Titre foncier | type de propriété: Autre | 1 b...,voursa.com,2026-01-20
3,987,دار للبيـــــــــــــــــــــع أفي عين الطلح,1600000.0,150.0,3.0,1.0,2.0,Teyarett,السلام عليكم ذاك دار للبيـــــــــــــــــــــ...,NaN,voursa.com,2025-12-09
4,252,ملح سكتير 2,800000.0,180.0,3.0,2.0,NaN,Toujounine,السلام عليكم \nفرصة للبيـــــــــــــــــــــع...,Titre foncier | 1 balcon(s) | Taille rue: 25.0...,voursa.com,2025-04-18


In [3]:
# Vérifier les valeurs manquantes
print("🔍 Valeurs manquantes :")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'N_Missing': missing, 'Pct_Missing': missing_pct})
missing_df = missing_df.query('N_Missing > 0').sort_values('Pct_Missing', ascending=False)
if len(missing_df) > 0:
    display(missing_df)
else:
    print("✅ Aucune valeur manquante")

🔍 Valeurs manquantes :


,N_Missing,Pct_Missing
nb_sdb,833,72.25
caracteristiques,157,13.62
nb_chambres,14,1.21


---
## 🔧 Section 3 : Feature Engineering Complet

### 3.1 Features de base

In [4]:
print("🔧 Feature Engineering - Features de base :")

# Prix au m²
if 'prix' in df.columns and 'surface_m2' in df.columns:
    df['prix_m2'] = df['prix'] / df['surface_m2']
    df.loc[df['surface_m2'] == 0, 'prix_m2'] = np.nan  # Éviter division par zéro
    print("  ✅ prix_m2 = prix / surface_m2")

# Log transform du prix
if 'prix' in df.columns:
    df['log_prix'] = np.log1p(df['prix'])
    print("  ✅ log_prix = log(1 + prix)")

# Nombre total de pièces
if 'nb_chambres' in df.columns and 'nb_salons' in df.columns:
    df['nb_pieces_total'] = df['nb_chambres'].fillna(0) + df['nb_salons'].fillna(0)
    print("  ✅ nb_pieces_total = nb_chambres + nb_salons")

print(f"\n📊 Nouvelles colonnes créées")
print(f"   Dimensions : {df.shape}")

🔧 Feature Engineering - Features de base :
  ✅ prix_m2 = prix / surface_m2
  ✅ log_prix = log(1 + prix)
  ✅ nb_pieces_total = nb_chambres + nb_salons

📊 Nouvelles colonnes créées
   Dimensions : (1153, 15)


In [5]:
# Parser les caractéristiques si pas déjà fait
def parse_caracteristiques(caracteristiques_str):
    """Parse la colonne caractéristiques et retourne un dictionnaire de features booléennes"""
    if pd.isna(caracteristiques_str) or caracteristiques_str == '':
        return {}
    
    features = {}
    caracteristiques_lower = str(caracteristiques_str).lower()
    
    # Features à détecter
    feature_keywords = {
        'has_garage': ['garage', 'garage'],
        'has_piscine': ['piscine', 'pool'],
        'has_clim': ['climatisation', 'clim', 'air conditionné'],
        'has_balcon': ['balcon'],
        'has_titre_foncier': ['titre foncier'],
        'has_camera': ['caméra', 'camera', 'sécurité'],
    }
    
    for feature_name, keywords in feature_keywords.items():
        features[feature_name] = any(keyword in caracteristiques_lower for keyword in keywords)
    
    return features

# Appliquer le parsing si la colonne existe
if 'caracteristiques' in df.columns:
    caracteristiques_parsed = df['caracteristiques'].apply(parse_caracteristiques)
    
    # Créer les colonnes booléennes
    for feature_name in ['has_garage', 'has_piscine', 'has_clim', 'has_balcon', 'has_titre_foncier', 'has_camera']:
        if feature_name not in df.columns:
            df[feature_name] = caracteristiques_parsed.apply(lambda x: x.get(feature_name, False))
    
    print("✅ Caractéristiques parsées")
    print(f"\n📊 Fréquence des caractéristiques :")
    for col in ['has_garage', 'has_piscine', 'has_clim', 'has_balcon', 'has_titre_foncier', 'has_camera']:
        if col in df.columns:
            print(f"  {col}: {df[col].sum()} ({df[col].mean()*100:.1f}%)")

✅ Caractéristiques parsées

📊 Fréquence des caractéristiques :
  has_garage: 472 (40.9%)
  has_piscine: 0 (0.0%)
  has_clim: 0 (0.0%)
  has_balcon: 796 (69.0%)
  has_titre_foncier: 480 (41.6%)
  has_camera: 97 (8.4%)


### 3.2 Feature age_annonce

In [6]:
# Calculer l'âge de l'annonce (nombre de jours depuis la publication)
if 'date_publication' in df.columns:
    # Convertir en datetime si ce n'est pas déjà fait
    df['date_publication'] = pd.to_datetime(df['date_publication'], errors='coerce')
    
    # Date de référence (aujourd'hui ou date la plus récente du dataset)
    date_ref = df['date_publication'].max() if df['date_publication'].notna().any() else datetime.now()
    
    # Calculer l'âge en jours
    df['age_annonce'] = (date_ref - df['date_publication']).dt.days
    df.loc[df['age_annonce'] < 0, 'age_annonce'] = 0  # Corriger les valeurs négatives
    
    print("✅ age_annonce créé (nombre de jours depuis la publication)")
    print(f"   Age moyen : {df['age_annonce'].mean():.1f} jours")
    print(f"   Age min : {df['age_annonce'].min():.0f} jours")
    print(f"   Age max : {df['age_annonce'].max():.0f} jours")
    
    # Remplacer les NaN par la médiane
    if df['age_annonce'].isnull().sum() > 0:
        median_age = df['age_annonce'].median()
        df['age_annonce'].fillna(median_age, inplace=True)
        print(f"   NaN remplacés par médiane ({median_age:.0f} jours)")
else:
    print("⚠️  Colonne 'date_publication' non trouvée")

✅ age_annonce créé (nombre de jours depuis la publication)
   Age moyen : 161.1 jours
   Age min : 0 jours
   Age max : 342 jours


### 3.3 Features additionnelles (interactions et ratios)

In [7]:
# Features d'interaction
if 'surface_m2' in df.columns and 'nb_chambres' in df.columns:
    df['surface_m2_x_nb_chambres'] = df['surface_m2'] * df['nb_chambres']
    print("  ✅ surface_m2_x_nb_chambres")

# Ratios
if 'nb_chambres' in df.columns and 'surface_m2' in df.columns:
    df['nb_chambres_per_m2'] = df['nb_chambres'] / (df['surface_m2'] + 1)  # +1 pour éviter division par zéro
    print("  ✅ nb_chambres_per_m2")

if 'nb_sdb' in df.columns and 'nb_chambres' in df.columns:
    df['nb_sdb_per_chambre'] = df['nb_sdb'] / (df['nb_chambres'] + 1)
    print("  ✅ nb_sdb_per_chambre")

print(f"\n📊 Dimensions finales : {df.shape}")

  ✅ surface_m2_x_nb_chambres
  ✅ nb_chambres_per_m2
  ✅ nb_sdb_per_chambre

📊 Dimensions finales : (1153, 25)


---
## 📊 Section 4 : Préparation des données

In [8]:
# 4.1 — Standardiser les quartiers (si pas déjà fait)
if 'quartier' in df.columns:
    quartier_mapping = {
        'Riyadh': 'Riyad',
        'Tevragh-Zeina': 'Tevragh Zeina',
        'TevraghZeina': 'Tevragh Zeina',
    }
    df['quartier'] = df['quartier'].replace(quartier_mapping)
    df['quartier'] = df['quartier'].str.title()
    print("✅ Quartiers standardisés")

✅ Quartiers standardisés


In [9]:
# 4.2 — Gestion des valeurs manquantes finales
print("🔧 Gestion des valeurs manquantes :")

# MCAR : médiane pour numériques
mcar_cols = ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb']
for col in mcar_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        med = df[col].median()
        n = df[col].isnull().sum()
        df[col].fillna(med, inplace=True)
        print(f"  {col}: {n} NaN → médiane ({med})")

# Remplacer les NaN dans les features booléennes par False
bool_cols = [col for col in df.columns if col.startswith('has_')]
for col in bool_cols:
    df[col] = df[col].fillna(False).astype(int)

print(f"\n✅ NaN restants : {df.isnull().sum().sum()}")

🔧 Gestion des valeurs manquantes :
  nb_chambres: 14 NaN → médiane (4.0)
  nb_sdb: 833 NaN → médiane (2.0)

✅ NaN restants : 1022


In [10]:
# 4.3 — Encodage des variables catégorielles
# One-Hot Encoding pour quartier et source
nominal_cols = [col for col in ['quartier', 'source'] if col in df.columns]

if len(nominal_cols) > 0:
    df_encoded = df.copy()
    
    for col in nominal_cols:
        # Garder top 10 catégories, regrouper le reste en "Autre"
        top_cats = df[col].value_counts().head(10).index
        df_encoded[col] = df[col].apply(lambda x: x if x in top_cats else 'Autre')
    
    # One-Hot Encoding
    df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=True, dtype=int)
    print(f"✅ One-Hot Encoding → {nominal_cols}")
    print(f"   Dimensions après encodage : {df_encoded.shape}")
    
    df = df_encoded

✅ One-Hot Encoding → ['quartier', 'source']
   Dimensions après encodage : (1153, 30)


In [11]:
# 4.4 — Sélection des features et préparation X, y
# Exclure les colonnes non pertinentes
exclude_cols = ['id', 'titre', 'description', 'caracteristiques', 'date_publication', 
                'prix', 'log_prix', 'prix_m2']  # prix_m2 et log_prix sont des features dérivées de y

# Sélectionner toutes les colonnes numériques et booléennes
feature_cols = [col for col in df.columns 
                if col not in exclude_cols 
                and df[col].dtype in ['int64', 'float64', 'bool']]

print(f"📊 Features sélectionnées : {len(feature_cols)}")
print(f"   Exemples : {feature_cols[:10]}")

# Préparer X et y
X = df[feature_cols].copy()
y = df['prix'].copy()

# Supprimer les lignes où y est NaN
mask = y.notna()
X = X[mask]
y = y[mask]

print(f"\n📐 Données finales :")
print(f"   X : {X.shape}")
print(f"   y : {y.shape}")
print(f"   Lignes avec prix valide : {mask.sum()}")

📊 Features sélectionnées : 22
   Exemples : ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb', 'nb_pieces_total', 'has_garage', 'has_piscine', 'has_clim', 'has_balcon', 'has_titre_foncier']

📐 Données finales :
   X : (1153, 22)
   y : (1153,)
   Lignes avec prix valide : 1153


In [12]:
# 4.5 — Split Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 Split Train/Test :")
print(f"   Train : {X_train.shape[0]} lignes ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"   Test  : {X_test.shape[0]} lignes ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"   Features : {len(feature_cols)}")

# Sauvegarder les noms des features
feature_names = list(X.columns)
print(f"\n✅ Split effectué avec random_state=42")

📊 Split Train/Test :
   Train : 922 lignes (80%)
   Test  : 231 lignes (20%)
   Features : 22

✅ Split effectué avec random_state=42


In [ ]:
# 4.6 — Vérifier et gérer les NaN avant normalisation
# #region agent log
import json
nan_cols = X_train.columns[X_train.isnull().any()].tolist()
nan_counts = {col: int(X_train[col].isnull().sum()) for col in nan_cols} if nan_cols else {}
with open('/home/bechir/Documents/immobilier-price-prediction/.cursor/debug.log', 'a') as f:
    f.write(json.dumps({"runId":"debug_run","hypothesisId":"B","location":"04_modeling.ipynb:Cell20","message":"NaN in X_train before scaling","data":{"nan_cols":nan_cols,"nan_counts":nan_counts,"total_nan":int(X_train.isnull().sum().sum())},"timestamp":int(__import__('time').time()*1000)}) + '\n')
# #endregion

# Remplacer les NaN restants par la médiane (pour chaque colonne)
if X_train.isnull().sum().sum() > 0:
    print(f"⚠️  {X_train.isnull().sum().sum()} NaN détectés dans X_train - imputation par médiane...")
    for col in X_train.columns:
        if X_train[col].isnull().sum() > 0:
            med = X_train[col].median()
            X_train[col].fillna(med, inplace=True)
            X_test[col].fillna(med, inplace=True)  # Utiliser la même médiane du train
    print("✅ NaN imputés")

# 4.7 — Normalisation (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit sur train seulement
X_test_scaled = scaler.transform(X_test)         # transform sur test

# #region agent log
import numpy as np
nan_in_scaled = np.isnan(X_train_scaled).sum()
with open('/home/bechir/Documents/immobilier-price-prediction/.cursor/debug.log', 'a') as f:
    f.write(json.dumps({"runId":"debug_run","hypothesisId":"B","location":"04_modeling.ipynb:Cell20","message":"NaN in X_train_scaled after scaling","data":{"nan_count":int(nan_in_scaled)},"timestamp":int(__import__('time').time()*1000)}) + '\n')
# #endregion

print("✅ StandardScaler appliqué")
print("   ⚠️  IMPORTANT : fit sur train, transform sur test (pas de data leakage !)")

✅ StandardScaler appliqué
   ⚠️  IMPORTANT : fit sur train, transform sur test (pas de data leakage !)


---
## 🤖 Section 5 : Modélisation

### 5.1 Définition des modèles à tester

In [14]:
# Définir les modèles à tester
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

# Ajouter XGBoost si disponible
if XGBOOST_AVAILABLE:
    models['XGBoost'] = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)

print(f"📊 {len(models)} modèles à tester :")
for name in models.keys():
    print(f"   - {name}")

📊 6 modèles à tester :
   - Linear Regression
   - Ridge
   - Lasso
   - Random Forest
   - Gradient Boosting
   - XGBoost


### 5.2 Validation croisée 5-fold

In [21]:
# Configuration de la validation croisée
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Métriques à calculer
scoring = {
    'neg_mse': 'neg_mean_squared_error',
    'neg_mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

print("🔄 Démarrage de la validation croisée 5-fold...")
print("   Cela peut prendre quelques minutes...\n")

results = {}

for name, model in models.items():
    print(f"📊 Évaluation de {name}...")
    
    # Validation croisée
    cv_results = cross_validate(
        model, 
        X_train_scaled, 
        y_train,
        cv=kfold,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    
    # Calculer RMSE et MAE (convertir de négatif à positif)
    rmse_scores = np.sqrt(-cv_results['test_neg_mse'])
    mae_scores = -cv_results['test_neg_mae']
    r2_scores = cv_results['test_r2']
    
    results[name] = {
        'RMSE_mean': rmse_scores.mean(),
        'RMSE_std': rmse_scores.std(),
        'MAE_mean': mae_scores.mean(),
        'MAE_std': mae_scores.std(),
        'R2_mean': r2_scores.mean(),
        'R2_std': r2_scores.std(),
    }
    
    print(f"   RMSE: {rmse_scores.mean():,.0f} ± {rmse_scores.std():,.0f} MRU")
    print(f"   MAE:  {mae_scores.mean():,.0f} ± {mae_scores.std():,.0f} MRU")
    print(f"   R²:   {r2_scores.mean():.4f} ± {r2_scores.std():.4f}\n")

print("✅ Validation croisée terminée !")

🔄 Démarrage de la validation croisée 5-fold...
   Cela peut prendre quelques minutes...

📊 Évaluation de Linear Regression...


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/home/bechir/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bechir/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/bechir/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py", line 618, in fit
    X, y = validate_data(
           ~~~~~~~~~~~~~^
        self,
        ^^^^^
    ...<5 lines>...
        force_writeable=True,
        ^^^^^^^^^^^^^^^^^^^^^
 

### 5.3 Tableau comparatif

In [ ]:
# Créer le DataFrame des résultats
results_df = pd.DataFrame(results).T
results_df = results_df.round(2)

# Réorganiser les colonnes pour meilleure lisibilité
results_df = results_df[['RMSE_mean', 'RMSE_std', 'MAE_mean', 'MAE_std', 'R2_mean', 'R2_std']]

print("📊 Résultats de la validation croisée (5-fold) :")
print("=" * 80)
display(results_df)

# Trier par R² (meilleur en premier)
results_df_sorted = results_df.sort_values('R2_mean', ascending=False)
print("\n🏆 Classement par R² :")
for i, (model, row) in enumerate(results_df_sorted.iterrows(), 1):
    print(f"   {i}. {model:20s} - R² = {row['R2_mean']:.4f} ± {row['R2_std']:.4f}")

In [ ]:
# Visualisation des résultats
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# RMSE
results_df_sorted['RMSE_mean'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('RMSE (MRU)')
axes[0].set_title('RMSE par modèle (plus bas = mieux)')
axes[0].invert_yaxis()

# MAE
results_df_sorted['MAE_mean'].plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('MAE (MRU)')
axes[1].set_title('MAE par modèle (plus bas = mieux)')
axes[1].invert_yaxis()

# R²
results_df_sorted['R2_mean'].plot(kind='barh', ax=axes[2], color='green')
axes[2].set_xlabel('R²')
axes[2].set_title('R² par modèle (plus haut = mieux)')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

---
## ✅ Section 6 : Sélection et évaluation du meilleur modèle

In [ ]:
# Identifier le meilleur modèle (basé sur R²)
best_model_name = results_df_sorted.index[0]
best_model = models[best_model_name]

print(f"🏆 Meilleur modèle : {best_model_name}")
print(f"   R² moyen (CV) : {results_df_sorted.loc[best_model_name, 'R2_mean']:.4f}")
print(f"   RMSE moyen (CV) : {results_df_sorted.loc[best_model_name, 'RMSE_mean']:,.0f} MRU")
print(f"\n🔄 Entraînement sur tout le train set...")

# Entraîner sur tout le train set
best_model.fit(X_train_scaled, y_train)

# Prédictions sur le test set
y_pred = best_model.predict(X_test_scaled)

# Métriques sur le test set
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
mae_test = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)

print(f"\n📊 Résultats sur le test set :")
print(f"   RMSE : {rmse_test:,.0f} MRU")
print(f"   MAE  : {mae_test:,.0f} MRU")
print(f"   R²   : {r2_test:.4f}")
print(f"   → Le modèle explique {r2_test*100:.1f}% de la variance de Prix")

In [ ]:
# Visualisation : Prédictions vs Réalité
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.4, s=15)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Prix réel (MRU)')
axes[0].set_ylabel('Prix prédit (MRU)')
axes[0].set_title(f'Prédictions vs Réalité (R²={r2_test:.3f})')
axes[0].grid(True, alpha=0.3)

# Distribution des résidus
residuals = y_test - y_pred
axes[1].hist(residuals, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Résidus (Prix réel - Prix prédit)')
axes[1].set_ylabel('Fréquence')
axes[1].set_title(f'Distribution des résidus (Moyenne={residuals.mean():,.0f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (pour Random Forest, Gradient Boosting, XGBoost)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Top 15 features
    top_features = feature_importance.head(15)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    top_features.plot(x='feature', y='importance', kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 15 Features - {best_model_name}')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Top 10 features les plus importantes :")
    display(top_features.head(10))
else:
    print("ℹ️  Feature importance non disponible pour ce type de modèle")

---
## 💾 Section 7 : Sauvegarde du modèle

In [ ]:
# Sauvegarder le modèle, le scaler et les noms des features
model_path = model_dir / 'housing_model.pkl'
scaler_path = model_dir / 'scaler.pkl'
features_path = model_dir / 'feature_names.pkl'

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(feature_names, features_path)

print("✅ Modèles sauvegardés :")
print(f"   📁 Modèle : {model_path}")
print(f"   📁 Scaler : {scaler_path}")
print(f"   📁 Features : {features_path}")
print(f"\n📊 Informations du modèle sauvegardé :")
print(f"   Nom : {best_model_name}")
print(f"   R² (test) : {r2_test:.4f}")
print(f"   RMSE (test) : {rmse_test:,.0f} MRU")

---
## 📝 Section 8 : Documentation

### Résumé des résultats

| Modèle | RMSE (CV) | MAE (CV) | R² (CV) | R² (Test) |
|--------|-----------|----------|---------|-----------|

In [ ]:
# Créer un tableau récapitulatif
summary_data = []
for name in results_df_sorted.index:
    cv_r2 = results_df_sorted.loc[name, 'R2_mean']
    cv_rmse = results_df_sorted.loc[name, 'RMSE_mean']
    cv_mae = results_df_sorted.loc[name, 'MAE_mean']
    
    # Si c'est le meilleur modèle, ajouter les résultats sur test
    if name == best_model_name:
        test_r2 = r2_test
        test_rmse = rmse_test
        test_mae = mae_test
    else:
        test_r2 = None
        test_rmse = None
        test_mae = None
    
    summary_data.append({
        'Modèle': name,
        'RMSE (CV)': f"{cv_rmse:,.0f}",
        'MAE (CV)': f"{cv_mae:,.0f}",
        'R² (CV)': f"{cv_r2:.4f}",
        'R² (Test)': f"{test_r2:.4f}" if test_r2 is not None else "-",
        'RMSE (Test)': f"{test_rmse:,.0f}" if test_rmse is not None else "-",
    })

summary_df = pd.DataFrame(summary_data)
print("📊 Tableau récapitulatif des résultats :")
display(summary_df)

### Choix du meilleur modèle

Le meilleur modèle est sélectionné automatiquement en fonction du R² moyen en validation croisée.

**Justification** :
- Meilleur R² moyen en validation croisée
- Performance stable (écart-type faible)
- Bonne généralisation sur le test set

### Features utilisées

- **Features numériques** : surface_m2, nb_chambres, nb_salons, nb_sdb, age_annonce
- **Features dérivées** : prix_m2, log_prix, nb_pieces_total, interactions, ratios
- **Features booléennes** : has_garage, has_piscine, has_clim, has_balcon, has_titre_foncier, has_camera
- **Features catégorielles encodées** : quartier (One-Hot), source (One-Hot)

### Limitations

- Modèles testés avec hyperparamètres par défaut (pas de grid search)
- Pas d'optimisation des hyperparamètres
- Features de texte (description, titre) non utilisées
- Données géographiques (si disponibles) non intégrées

### Améliorations futures

1. **Optimisation des hyperparamètres** : Grid Search ou Random Search
2. **Feature Engineering avancé** : 
   - Analyse NLP des descriptions
   - Features géographiques (distances, POI)
   - Features temporelles (saisonnalité)
3. **Ensemble methods** : Stacking, Blending
4. **Sélection de features** : RFE, Lasso pour réduire la dimensionnalité
5. **Gestion des outliers** : Techniques robustes ou transformation des variables

---
*🎓 Notebook créé pour le projet Capstone - Prédiction des Prix Immobiliers en Mauritanie*  
*📅 SupNum - Machine Learning Course - Mohamed*  
*Février 2026*